# Free-$\beta$ perturbation search for the tripartite cycle polytope

## Purpose

This notebook performs the missing combination of two earlier calculations:

1. the ordered-pair perturbation search of `Cycle_Perturbation_Search.ipynb`;
2. the free-$\beta$, white-noise-normalized separator LP of
   `Cycle_White_Noise_Separator_LP.ipynb`.

For each nonsignalling extremal representative
$q\in\{q_{27},q_{29},q_{43},q_{44}\}$ and each ordered pair of distinct
siglocal vertices $(v_j,v_k)$, it constructs

$$
q_{jk}^{(\epsilon)}=
\left(1-\frac{3\epsilon}{2}\right)q
+\epsilon v_j+\frac{\epsilon}{2}v_k,
\qquad \epsilon=10^{-3},
$$

and solves

$$
\begin{aligned}
\operatorname*{minimize}_{c,\beta}\quad
    &c\cdot q_{jk}^{(\epsilon)}-\beta\\
\text{subject to}\quad
    &c\cdot v_i\geq\beta
      \qquad i=1,\ldots,8128,\\
    &c\cdot u-\beta=1,
\end{aligned}
$$

where $u(abc\mid xyz)=1/8$. All 64 coefficients of $c$ and $\beta$ are
unrestricted variables. This is the same direct 64-probability-coordinate
formulation used in the earlier free-$\beta$ notebook; no affine-gauge
projection is introduced here.

The siglocal polytope has affine dimension 50 inside $\mathbb R^{64}$.
Consequently, different pairs $(c,\beta)$ can define exactly the same affine
functional on every behavior in its affine hull. The notebook deliberately
accepts this harmless redundancy: it removes only identical, exactly
reconstructed coefficient tuples. Gauge-equivalent tuples may therefore be
retained as separate records. This does not affect validity, target slack,
white-noise normalization, or the facet test.

Every retained coefficient vector is audited exactly on all 8128 vertices.
Nonfacets are not discarded: the program records the exact dimension of the
saturated face and labels a candidate as a facet precisely when that
dimension is 49.

This notebook generates inequalities only. NPA screening and see-saw
optimization belong in later notebooks.


## How to run

1. Open the notebook with the Julia kernel.
2. Choose **Kernel → Restart Kernel and Run All Cells**.
3. The package-bootstrap cell installs only packages that are missing from the
   active Julia environment.
4. The final configuration initially uses **60 seconds per class** as a safe
   installation and logic test. Increase `TARGET_SECONDS_PER_CLASS` only after
   checking that this short run finishes normally.

Results are written to `cycle_free_beta_perturbation_results` beside the
notebook. Every class has a machine-readable checkpoint, a human-readable TSV
table, and a failure log. Rerunning the final search cell resumes from the next
ordered pair and uses only the unspent part of the requested cumulative time.

The complete scan contains 66,056,256 ordered pairs per class and is not
expected to finish during a short local run.


## 1. Canonical probability coordinates

A behavior in the binary-input, binary-output tripartite scenario consists of

$$
p(abc\mid xyz),
\qquad a,b,c,x,y,z\in\{0,1\}.
$$

It is stored as a vector of length $2^6=64$. To remove any ambiguity, the vector is split into eight input blocks, ordered lexicographically as

$$
xyz=000,001,010,011,100,101,110,111.
$$

Inside each input block, the eight outputs use the same order,

$$
abc=000,001,010,011,100,101,110,111.
$$

Thus $z$ is the fastest-changing input bit and $c$ is the fastest-changing output bit. In Julia's one-based indexing, the coordinate of $p(abc\mid xyz)$ is

$$
1+c+2b+4a+8z+16y+32x.
$$

The binary outputs are also interpreted as signs when correlators are used later: output `0` corresponds to $+1$, and output `1` corresponds to $-1$.

The two assertions at the end of the next cell are small but important regression tests. They verify both the within-block output order and the order of the eight input blocks. Keeping this map explicit prevents the kind of silent iterator-order error that can change a polytope while leaving array dimensions apparently correct.


In [4]:
# Package bootstrap. Existing packages are not changed; only missing packages
# are installed in the active Julia environment.
import Pkg

required_packages = [
    "JuMP",
    "MathOptInterface",
    "GLPK",
]

missing_packages = [
    package for package in required_packages
    if Base.find_package(package) === nothing
]

if !isempty(missing_packages)
    println("Installing missing Julia packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required Julia packages are already available.")
end


All required Julia packages are already available.


In [5]:
using LinearAlgebra
using Printf
using JuMP
using GLPK

# Import MathOptInterface directly. In particular, do not use
# `JuMP.MathOptInterface`, which caused an error with the user's Julia 1.12
# installation.
import MathOptInterface as MOI

const CLASSES = (27, 29, 43, 44)
const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

#
# The 64-vector is ordered in blocks of fixed (x,y,z):
#
#   (x,y,z) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Within each block, the output order is
#
#   (a,b,c) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Thus c and z are the fastest-changing indices. Binary output 0 corresponds
# to the physicists' value +1, and binary output 1 corresponds to -1.

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)


## 2. Correct deterministic vertices of the two cycle orientations

Each party uses a deterministic binary response function of two binary inputs. There are

$$
2^{2^2}=16
$$

such functions. The integer masks `0:15` encode their four-entry truth tables; `response(mask,u,v)` reads the output assigned to the pair $(u,v)$.

The two communication orientations have different causal dependencies:

| orientation | Alice | Bob | Charlie |
|---|---|---|---|
| clockwise | $a=f_A(x,z)$ | $b=f_B(y,x)$ | $c=f_C(z,y)$ |
| counterclockwise | $a=f_A(x,y)$ | $b=f_B(y,z)$ | $c=f_C(z,x)$ |

Choosing one of 16 response functions independently for each party gives $16^3=4096$ deterministic behaviors per orientation. Every deterministic behavior has exactly one probability equal to 1 in each input block and all other entries equal to 0.

The polytope used by the LP is the convex hull of the **union** of the two orientation lists. Their intersection contains the 64 fully local deterministic behaviors, so removing duplicate rows gives

$$
4096+4096-64=8128
$$

distinct vertices. Removing duplicates does not change the convex hull or any LP optimum, but it exposes the correct combinatorics and avoids redundant constraints.

This cell performs structural checks before any inequality is computed: binary entries, one selected output per input, expected vertex counts, intersection size, and affine dimensions. The individual orientation polytopes have affine dimension 44; their corrected union has affine dimension 50. In particular, the jump from 44 to 50 is a strong diagnostic that the second orientation was not accidentally generated as a copy of the first.


In [7]:

"""
    response(mask, u, v)

Evaluate one of the 16 deterministic binary functions of two binary inputs.
Bit `2u + v` of `mask` stores the output for the input pair `(u,v)`.
"""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""
    deterministic_vertex(fA, fB, fC, orientation)

Construct one deterministic 64-component behavior in canonical coordinates.

For `:clockwise`:
    a = fA(x,z),  b = fB(y,x),  c = fC(z,y).

For `:counterclockwise`:
    a = fA(x,y),  b = fB(y,z),  c = fC(z,x).
"""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one cycle orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function affine_dimension(points; atol=1e-9)
    n = size(points, 1)
    n == 0 && return -1
    n == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64

    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)

        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end

    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# `unique(...; dims=1)` removes the 64 local deterministic behaviors that occur
# in both orientations. Keeping duplicates would not change the LP, but it would
# obscure the actual vertex count and the geometry.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle, 1) == 8128

@assert affine_dimension(vertices_clockwise) == 44
@assert affine_dimension(vertices_counterclockwise) == 44
@assert affine_dimension(vertices_cycle) == 50

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", affine_dimension(vertices_cycle))




Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## 3. Published representatives of NS classes 27, 29, 43, and 44

The four targets are entered through the 26 correlators of a binary tripartite nonsignalling behavior. Their stored order is

$$
\begin{aligned}
&(A_0,A_1;\ B_0,B_1;\ C_0,C_1;\\
&AB_{00},AB_{01},AB_{10},AB_{11};\\
&AC_{00},AC_{01},AC_{10},AC_{11};\\
&BC_{00},BC_{01},BC_{10},BC_{11};\\
&ABC_{000},ABC_{001},ABC_{010},ABC_{011},
ABC_{100},ABC_{101},ABC_{110},ABC_{111}).
\end{aligned}
$$

These representatives are those reported in Table 1 of S. Pironio, J.-D. Bancal, and V. Scarani, *Extremal correlations of the tripartite no-signaling polytope*, J. Phys. A **44**, 065303 (2011), arXiv:1101.2477.

To convert correlators into probabilities, define $s_a=(-1)^a$, and analogously for $b$ and $c$. Then

$$
\begin{aligned}
p(abc\mid xyz)=\frac18[&1+s_aA_x+s_bB_y+s_cC_z
+s_as_bAB_{xy}+s_as_cAC_{xz}\\
&+s_bs_cBC_{yz}+s_as_bs_cABC_{xyz}].
\end{aligned}
$$

The implementation uses Julia rational numbers such as `1//3` throughout this conversion. Consequently, positivity, normalization, nonsignalling, and the later evaluations of rationalized inequalities can be checked exactly rather than up to floating-point tolerance.


In [9]:
#
# Correlator order:
#
#   A0,A1, B0,B1, C0,C1,
#   AB00,AB01,AB10,AB11,
#   AC00,AC01,AC10,AC11,
#   BC00,BC01,BC10,BC11,
#   ABC000,ABC001,ABC010,ABC011,ABC100,ABC101,ABC110,ABC111.
#
# These representatives agree with Table 1 of:
# S. Pironio, J.-D. Bancal, and V. Scarani,
# "Extremal correlations of the tripartite no-signaling polytope",
# J. Phys. A 44, 065303 (2011), arXiv:1101.2477.

const NS_CORRELATORS = Dict(
    27 => Rational{Int}[
        1//2, 0, 0, 0, 0, 0,
        1//2, 1//2, 0, 0,
        1//2, 1//2, 0, 0,
        0, 0, 0, 0,
        -1//2, -1//2, -1//2, -1//2, 1, 1, 1, -1,
    ],
    29 => Rational{Int}[
        1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
    ],
    43 => Rational{Int}[
        1//7, 1//7, 1//7, 1//7, 1//7, 1//7,
        3//7, -1//7, -1//7, -1//7,
        3//7, -1//7, -1//7, -1//7,
        -1//7, -1//7, -1//7, -1//7,
        1//7, 5//7, 5//7, -1, -1, -1, -1, -1,
    ],
    44 => Rational{Int}[
        0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, -1,
    ],
)

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

"""Convert the 26 NS correlators to the canonical 64 probabilities exactly."""
function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)

        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] +
            sb*B[y + 1] +
            sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end

    return probabilities
end

const NS_TARGETS = Dict(
    class => correlators_to_probabilities(NS_CORRELATORS[class])
    for class in CLASSES
)




Dict{Int64, Vector{Rational{Int64}}} with 4 entries:
  29 => [2//3, 0, 0, 0, 0, 0, 0, 1//3, 1//3, 1//3  …  0, 0, 0, 1//3, 1//3, 0, 1…
  43 => [2//7, 1//7, 1//7, 0, 0, 1//7, 1//7, 1//7, 2//7, 1//7  …  0, 1//7, 0, 2…
  27 => [1//4, 1//4, 1//4, 0, 0, 0, 0, 1//4, 1//4, 1//4  …  1//4, 0, 0, 1//4, 1…
  44 => [1//4, 0, 0, 1//4, 0, 1//4, 1//4, 0, 1//4, 0  …  1//4, 0, 0, 1//4, 1//4…

## 4. Exact target checks: positivity, normalization, nonsignalling, and extremality

Before using a target in an LP, this cell verifies the defining probabilistic and nonsignalling conditions directly in the 64-coordinate representation:

- every probability is nonnegative;
- the eight probabilities for each fixed input triple $(x,y,z)$ sum to 1;
- the $AB$ marginal is independent of Charlie's input $z$;
- the $AC$ marginal is independent of Bob's input $y$;
- the $BC$ marginal is independent of Alice's input $x$.

It then independently checks that each representative is a **vertex** of the nonsignalling polytope. The normalization and nonsignalling equalities are assembled as a (redundant) $56\times64$ matrix. Its rank is 38, hence the nonsignalling affine space has dimension

$$
64-38=26.
$$

At a given target, every zero probability saturates one positivity inequality. The target is extremal when the equality rows together with these active positivity rows have rank 64: there is then no nonzero feasible direction that keeps all active constraints saturated. The printed `active rank = 64 (NS vertex)` is therefore a geometric check, not merely a lookup of the class label.

All these tests use exact target probabilities. A failed assertion indicates an inconsistent representative, correlator order, probability order, or nonsignalling implementation and should be investigated before proceeding.


In [11]:

function validate_nonsignalling_behavior(probabilities)
    @assert length(probabilities) == 64
    @assert all(probability -> probability >= 0, probabilities)

    # Normalization for every input triple.
    for (x, y, z) in TRIPLES
        @assert sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        ) == 1
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        @assert lhs == rhs
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        @assert lhs == rhs
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        @assert lhs == rhs
    end

    return true
end

"""
Return a (redundant) matrix for normalization and tripartite NS equalities.
Its rank must be 38, leaving the familiar 26-dimensional NS affine space.
"""
function nonsignalling_equality_matrix()
    equalities = zeros(Float64, 56, 64)
    row = 0

    for (x, y, z) in TRIPLES
        row += 1
        for (a, b, c) in TRIPLES
            equalities[row, prob_index(a, b, c, x, y, z)] = 1
        end
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        row += 1
        for c in 0:1
            equalities[row, prob_index(a, b, c, x, y, 0)] += 1
            equalities[row, prob_index(a, b, c, x, y, 1)] -= 1
        end
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        row += 1
        for b in 0:1
            equalities[row, prob_index(a, b, c, x, 0, z)] += 1
            equalities[row, prob_index(a, b, c, x, 1, z)] -= 1
        end
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        row += 1
        for a in 0:1
            equalities[row, prob_index(a, b, c, 0, y, z)] += 1
            equalities[row, prob_index(a, b, c, 1, y, z)] -= 1
        end
    end

    @assert row == 56
    return equalities
end

const NS_EQUALITIES = nonsignalling_equality_matrix()
@assert rank(NS_EQUALITIES; atol=1e-9, rtol=0.0) == 38

"""
Check extremality in the NS polytope using the rank of the NS equalities plus
the positivity inequalities saturated by the target.
"""
function ns_active_rank(probabilities)
    zero_coordinates = findall(iszero, probabilities)
    coordinate_rows = Matrix{Float64}(I, 64, 64)[zero_coordinates, :]
    active_matrix = vcat(NS_EQUALITIES, coordinate_rows)
    return rank(active_matrix; atol=1e-9, rtol=0.0)
end

println("\nTarget checks")
for class in CLASSES
    target = NS_TARGETS[class]
    validate_nonsignalling_behavior(target)
    active_rank = ns_active_rank(target)
    @assert active_rank == 64

    @printf(
        "  class %2d: support = %2d, active rank = %2d (NS vertex)\n",
        class,
        count(value -> !iszero(value), target),
        active_rank,
    )
end





Target checks
  class 27: support = 32, active rank = 64 (NS vertex)
  class 29: support = 23, active rank = 64 (NS vertex)
  class 43: support = 36, active rank = 64 (NS vertex)
  class 44: support = 32, active rank = 64 (NS vertex)


## 5. Exact affine-dimension audit

The free-$\beta$ LP below is solved directly in all 64 canonical probability
coordinates. There is no coordinate selection, projection, or canonical
lifting step.

Facetness nevertheless requires an exact geometric calculation. The 8128
vertices span an affine space of dimension 50, and a valid inequality is a
facet exactly when its saturating vertices have affine dimension 49.

The following rank routine is inherited from the earlier perturbation
notebook. It computes ranks modulo several primes and then uses a Hadamard
bound to certify that the modular result equals the rational rank. This code
is used only for the affine dimension and later face-dimension tests; it
does not alter the LP variables or objective.


In [13]:
using Dates
using Serialization

# Pairwise-distinct primes that fit in Int64. Products are evaluated through
# Int128, preventing overflow in modular multiplication.
const RANK_PRIMES = Int64[
    1_000_000_007,
    1_000_000_009,
    998_244_353,
    1_004_535_809,
]

mod_multiply(a::Int64, b::Int64, modulus::Int64) =
    Int64(mod(Int128(a) * Int128(b), Int128(modulus)))

function mod_power(base::Int64, exponent::Int64, modulus::Int64)
    result = Int64(1)
    factor = mod(base, modulus)
    power = exponent

    while power > 0
        if isodd(power)
            result = mod_multiply(result, factor, modulus)
        end
        factor = mod_multiply(factor, factor, modulus)
        power >>= 1
    end

    return result
end

"""Compute the rank of an integer matrix over the field Z/modulus*Z."""
function rank_mod_prime(
    matrix::AbstractMatrix{<:Integer},
    modulus::Int64,
)
    n_rows, n_columns = size(matrix)
    work = Matrix{Int64}(undef, n_rows, n_columns)

    @inbounds for index in eachindex(matrix)
        work[index] = mod(Int64(matrix[index]), modulus)
    end

    pivot_row = 1

    @inbounds for column in 1:n_columns
        pivot_row > n_rows && break

        pivot = 0
        for row in pivot_row:n_rows
            if work[row, column] != 0
                pivot = row
                break
            end
        end
        pivot == 0 && continue

        if pivot != pivot_row
            for j in 1:n_columns
                work[pivot_row, j], work[pivot, j] =
                    work[pivot, j], work[pivot_row, j]
            end
        end

        inverse_pivot = mod_power(
            work[pivot_row, column],
            modulus - 2,
            modulus,
        )

        for row in (pivot_row + 1):n_rows
            entry = work[row, column]
            entry == 0 && continue

            factor = mod_multiply(entry, inverse_pivot, modulus)
            for j in column:n_columns
                product = mod_multiply(
                    factor,
                    work[pivot_row, j],
                    modulus,
                )
                work[row, j] = Int64(mod(
                    Int128(work[row, j]) - Int128(product),
                    Int128(modulus),
                ))
            end
        end

        pivot_row += 1
    end

    return pivot_row - 1
end

"""
    certified_integer_rank(matrix)

Return the exact rational rank of an integer matrix.

A modular rank is a lower bound on the rational rank. If the largest
modular rank is `r`, every `(r+1)`-minor vanishes modulo every listed prime.
The Hadamard estimate below proves that such a minor is too small in
absolute value to be a nonzero multiple of their product.
"""
function certified_integer_rank(matrix::AbstractMatrix{<:Integer})
    n_rows, n_columns = size(matrix)
    min(n_rows, n_columns) == 0 && return 0

    ranks = [rank_mod_prime(matrix, prime) for prime in RANK_PRIMES]
    certified_rank = maximum(ranks)

    # Full row or column rank is exact once witnessed modulo one prime.
    certified_rank == min(n_rows, n_columns) && return certified_rank

    maximum_squared_row_norm = maximum(
        sum(BigInt(matrix[row, column])^2 for column in 1:n_columns)
        for row in 1:n_rows
    )
    maximum_squared_row_norm == 0 && return 0

    next_minor_size = certified_rank + 1
    modulus_product = prod(BigInt(prime) for prime in RANK_PRIMES)
    hadamard_squared_bound = maximum_squared_row_norm^next_minor_size

    modulus_product^2 > hadamard_squared_bound || error(
        "The modular primes are insufficient to certify this rank exactly.",
    )

    return certified_rank
end

"""Compute the exact affine dimension of a set of integer points."""
function certified_affine_dimension(points::AbstractMatrix{<:Integer})
    n_points, dimension = size(points)
    n_points == 0 && return -1
    n_points == 1 && return 0

    differences = Matrix{Int64}(undef, n_points - 1, dimension)
    @inbounds for row in 2:n_points, column in 1:dimension
        differences[row - 1, column] =
            Int64(points[row, column]) - Int64(points[1, column])
    end

    return certified_integer_rank(differences)
end

const SIGLOCAL_AFFINE_DIMENSION = certified_affine_dimension(vertices_cycle)
@assert SIGLOCAL_AFFINE_DIMENSION == 50

println("\nExact geometry audit")
println("  probability coordinates:      64")
println("  LP coefficient coordinates:   64 (no affine gauge fixing)")
println("  siglocal affine dimension:    ", SIGLOCAL_AFFINE_DIMENSION)



Exact geometry audit
  probability coordinates:      64
  LP coefficient coordinates:   64 (no affine gauge fixing)
  siglocal affine dimension:    50


## 6. Direct 64-coordinate free-$\beta$ LP and exact candidate audit

The uniform behavior is first verified to be exactly the average of the
8128 distinct siglocal vertices. The LP model is then built once using all
64 probability coefficients. During the search, only the objective
coefficients change.

For every numerical LP output, the code:

1. reconstructs 64 rational coefficients and a rational $\beta$;
2. renormalizes them exactly so that $c\cdot u-\beta=1$;
3. verifies validity on all 8128 vertices;
4. checks agreement between the exact perturbed-target slack and the
   numerical LP optimum;
5. identifies all saturating vertices and certifies their affine dimension;
6. saves both the normalized rational form and a primitive integer form.

Distinctness is tested on the exact 64 coefficients together with $\beta$.
There is no rounded floating-point prefilter. Because no affine gauge is
fixed, two different stored tuples may still represent the same functional
on the 50-dimensional affine hull; this is an intentional and harmless
consequence of using the simpler original LP.


In [15]:
const UNIFORM_BEHAVIOUR = fill(1//8, 64)

# Each probability coordinate occurs in exactly 1016 of the 8128 vertices.
# Therefore the average of all distinct vertices is exactly white noise.
@assert all(sum(vertices_cycle; dims=1) .== 1016)
@assert [
    sum(Int(vertices_cycle[i, j]) for i in axes(vertices_cycle, 1)) //
    size(vertices_cycle, 1)
    for j in axes(vertices_cycle, 2)
] == UNIFORM_BEHAVIOUR

"""Rationalize one numerical value, treating numerical zeros explicitly."""
function rationalize_scalar(value; tol=1e-8)
    return abs(value) <= tol ?
        BigInt(0)//BigInt(1) :
        rationalize(BigInt, value; tol=tol)
end

"""Convert a rational inequality to a primitive integer representative."""
function primitive_integer_form(coefficients, beta)
    all_values = vcat(coefficients, [beta])
    common_denominator = foldl(
        lcm,
        denominator.(all_values);
        init=BigInt(1),
    )
    integer_values = BigInt[
        numerator(value * common_denominator)
        for value in all_values
    ]

    common_divisor = foldl(
        gcd,
        abs.(integer_values);
        init=BigInt(0),
    )
    common_divisor == 0 && error("The reconstructed inequality is zero")
    integer_values .÷= common_divisor

    return integer_values[1:end-1], integer_values[end]
end

"""Evaluate one inequality exactly on every supplied vertex."""
function exact_vertex_slacks(vertices, coefficients, beta)
    n_vertices, dimension = size(vertices)
    @assert dimension == length(coefficients) == 64

    return Rational{BigInt}[
        sum(
            coefficients[column] * Int(vertices[row, column])
            for column in 1:dimension
        ) - beta
        for row in 1:n_vertices
    ]
end

exact_behavior_slack(behavior, coefficients, beta) =
    sum(coefficients .* behavior) - beta

"""
Build the unrestricted 64-coordinate free-beta model once. Only the
objective changes between targets.
"""
function build_free_beta_model(vertices, uniform_behavior)
    number_of_vertices, dimension = size(vertices)
    @assert dimension == length(uniform_behavior) == 64

    model = Model(GLPK.Optimizer)
    set_silent(model)

    # JuMP variables are unrestricted unless bounds are explicitly added.
    @variable(model, coefficients[1:dimension])
    @variable(model, beta)

    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= beta,
    )

    @constraint(
        model,
        white_noise_normalization,
        sum(
            Float64(uniform_behavior[j]) * coefficients[j]
            for j in 1:dimension
        ) - beta == 1,
    )

    # A linear objective containing every variable is installed before its
    # coefficients are updated for the first target.
    @objective(model, Min, sum(coefficients) - beta)

    return model, coefficients, beta
end

"""Replace the objective and solve the reusable free-beta LP."""
function solve_free_beta_target!(model, coefficients, beta, target)
    @assert length(coefficients) == length(target) == 64

    set_objective_coefficient(model, coefficients, Float64.(target))
    set_objective_coefficient(model, beta, -1.0)
    optimize!(model)

    status = termination_status(model)
    status == MOI.OPTIMAL || return nothing, status, NaN

    solution = (
        coefficients=value.(coefficients),
        beta=value(beta),
    )
    return solution, status, objective_value(model)
end

"""
Rationally reconstruct and exactly white-noise-normalize an LP solution.

The normalization removes the overall positive scale. It intentionally
does not choose a representative of the remaining affine-gauge freedom.
"""
function reconstruct_free_beta_solution(
    coefficients_float,
    beta_float;
    rational_tolerance=1e-8,
)
    all(isfinite, coefficients_float) || return nothing
    isfinite(beta_float) || return nothing

    coefficients_rational = Rational{BigInt}[
        rationalize_scalar(value; tol=rational_tolerance)
        for value in coefficients_float
    ]
    beta_rational = rationalize_scalar(
        beta_float;
        tol=rational_tolerance,
    )

    uniform_slack = exact_behavior_slack(
        UNIFORM_BEHAVIOUR,
        coefficients_rational,
        beta_rational,
    )
    uniform_slack > 0 || return nothing

    exact_coefficients = coefficients_rational ./ uniform_slack
    exact_beta = beta_rational / uniform_slack

    @assert exact_behavior_slack(
        UNIFORM_BEHAVIOUR,
        exact_coefficients,
        exact_beta,
    ) == 1

    return (
        coefficients=exact_coefficients,
        beta=exact_beta,
    )
end

candidate_key(coefficients, beta) = Tuple(vcat(coefficients, [beta]))

function critical_values(slack)
    if slack < 0
        visibility = 1 / (1 - slack)
        noise_fraction = -slack / (1 - slack)
        return visibility, noise_fraction
    end
    return nothing, nothing
end

"""Turn one reconstructed solution into a completely audited record."""
function audit_free_beta_candidate(
    source::Symbol,
    class::Int,
    pair_j::Int,
    pair_k::Int,
    reconstructed,
    solver_objective,
    original_target,
    perturbed_target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    objective_tolerance=1e-7,
)
    exact_coefficients = reconstructed.coefficients
    exact_beta = reconstructed.beta

    cycle_slacks = exact_vertex_slacks(
        vertices_cycle,
        exact_coefficients,
        exact_beta,
    )
    clockwise_slacks = exact_vertex_slacks(
        vertices_clockwise,
        exact_coefficients,
        exact_beta,
    )
    counterclockwise_slacks = exact_vertex_slacks(
        vertices_counterclockwise,
        exact_coefficients,
        exact_beta,
    )

    minimum(cycle_slacks) >= 0 || error(
        "Rational reconstruction produced an invalid siglocal inequality",
    )

    uniform_slack = exact_behavior_slack(
        UNIFORM_BEHAVIOUR,
        exact_coefficients,
        exact_beta,
    )
    uniform_slack == 1 || error(
        "Exact white-noise normalization failed",
    )

    original_target_slack = exact_behavior_slack(
        original_target,
        exact_coefficients,
        exact_beta,
    )
    perturbed_target_slack = exact_behavior_slack(
        perturbed_target,
        exact_coefficients,
        exact_beta,
    )

    numerical_disagreement = abs(
        Float64(perturbed_target_slack) - solver_objective,
    )
    allowed_disagreement = objective_tolerance * (1 + abs(solver_objective))
    numerical_disagreement <= allowed_disagreement || error(
        "Exact and numerical perturbed-target objectives disagree",
    )

    saturating_indices = findall(iszero, cycle_slacks)
    isempty(saturating_indices) && error(
        "The reconstructed inequality does not touch the siglocal polytope",
    )
    face_dimension = certified_affine_dimension(
        vertices_cycle[saturating_indices, :],
    )

    primitive_coefficients, primitive_beta = primitive_integer_form(
        exact_coefficients,
        exact_beta,
    )

    original_visibility, original_noise_fraction =
        critical_values(original_target_slack)
    perturbed_visibility, perturbed_noise_fraction =
        critical_values(perturbed_target_slack)

    return (
        source=source,
        class=class,
        first_j=pair_j,
        first_k=pair_k,
        exact_coefficients=exact_coefficients,
        exact_beta=exact_beta,
        primitive_coefficients=primitive_coefficients,
        primitive_beta=primitive_beta,
        solver_objective=solver_objective,
        original_target_slack=original_target_slack,
        perturbed_target_slack=perturbed_target_slack,
        separates_original=(original_target_slack < 0),
        separates_perturbed=(perturbed_target_slack < 0),
        original_critical_visibility=original_visibility,
        original_critical_noise_fraction=original_noise_fraction,
        perturbed_critical_visibility=perturbed_visibility,
        perturbed_critical_noise_fraction=perturbed_noise_fraction,
        exact_uniform_slack=uniform_slack,
        exact_cycle_minimum_slack=minimum(cycle_slacks),
        exact_clockwise_minimum_slack=minimum(clockwise_slacks),
        exact_counterclockwise_minimum_slack=minimum(counterclockwise_slacks),
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=(face_dimension == SIGLOCAL_AFFINE_DIMENSION - 1),
    )
end

free_beta_model, free_beta_coefficients, free_beta_beta =
    build_free_beta_model(vertices_cycle, UNIFORM_BEHAVIOUR)


(A JuMP Model
├ solver: GLPK
├ objective_sense: MIN_SENSE
│ └ objective_function_type: AffExpr
├ num_variables: 65
├ num_constraints: 8129
│ ├ AffExpr in MOI.EqualTo{Float64}: 1
│ └ AffExpr in MOI.GreaterThan{Float64}: 8128
└ Names registered in the model
  └ :beta, :coefficients, :valid_on_cycle, :white_noise_normalization, VariableRef[coefficients[1], coefficients[2], coefficients[3], coefficients[4], coefficients[5], coefficients[6], coefficients[7], coefficients[8], coefficients[9], coefficients[10]  …  coefficients[55], coefficients[56], coefficients[57], coefficients[58], coefficients[59], coefficients[60], coefficients[61], coefficients[62], coefficients[63], coefficients[64]], beta)

## 7. Unperturbed regression test

Before the ordered-pair search begins, the reusable 64-coordinate LP is
applied to the four original NS representatives. These are the free-$\beta$
calculations already performed in the earlier separator notebook.

The exact optimal slacks must be

$$
\delta_{27}=-\frac1{11},\qquad
\delta_{29}=-\frac8{45},\qquad
\delta_{43}=-\frac2{21},\qquad
\delta_{44}=-\frac13.
$$

Reproducing these four values checks the vertex set, targets, direct
64-coordinate LP constraints, white-noise normalization, objective update,
and rational reconstruction. Each unperturbed inequality is also inserted
as the first candidate of its class, with pair indices `(0,0)`.


In [17]:
const EXPECTED_TARGET_SLACKS = Dict(
    27 => -1//11,
    29 => -8//45,
    43 => -2//21,
    44 => -1//3,
)

const BASELINE_RESULTS = Dict{Int, Any}()

for class in CLASSES
    target = NS_TARGETS[class]
    numerical_solution, status, solver_objective = solve_free_beta_target!(
        free_beta_model,
        free_beta_coefficients,
        free_beta_beta,
        target,
    )
    status == MOI.OPTIMAL || error(
        "Unperturbed free-beta LP for class $class ended with status $status",
    )

    reconstructed = reconstruct_free_beta_solution(
        numerical_solution.coefficients,
        numerical_solution.beta,
    )
    reconstructed === nothing && error(
        "Could not rationally reconstruct the class-$class baseline",
    )

    candidate = audit_free_beta_candidate(
        :unperturbed,
        class,
        0,
        0,
        reconstructed,
        solver_objective,
        target,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    candidate.original_target_slack == EXPECTED_TARGET_SLACKS[class] || error(
        "Baseline regression failed for class $class: obtained " *
        "$(candidate.original_target_slack), expected " *
        "$(EXPECTED_TARGET_SLACKS[class])",
    )
    BASELINE_RESULTS[class] = candidate
end

println("\nUnperturbed free-beta regression test")
println("class  target-slack  visibility  noise-frac  saturating  face-dim  facet")
for class in CLASSES
    candidate = BASELINE_RESULTS[class]
    @printf(
        "%5d  %12s  %10s  %10s  %10d  %8d  %s\n",
        class,
        string(candidate.original_target_slack),
        string(candidate.original_critical_visibility),
        string(candidate.original_critical_noise_fraction),
        candidate.saturation_count,
        candidate.face_dimension,
        string(candidate.is_facet),
    )
end



Unperturbed free-beta regression test
class  target-slack  visibility  noise-frac  saturating  face-dim  facet
   27        -1//11      11//12       1//12         119        49  true
   29        -8//45      45//53       8//53         270        49  true
   43        -2//21      21//23       2//23         508        49  true
   44         -1//3        3//4        1//4         508        49  true


# Stage 8: time-limited ordered-pair perturbation search

For every class, the deterministic search order is

```text
for j in 1:8128
    for k in 1:8128
        j == k && continue
        ...
    end
end
```

The two weights are unequal, so $(j,k)$ and $(k,j)$ are different objective
directions. The vertices may signal in the ordinary Bell sense; the perturbed
point is only an auxiliary LP target and is not claimed to be a physical
behavior.

The complete scan has

$$
8128\times8127=66{,}056{,}256
$$

LPs per class. The local version therefore uses a cumulative wall-clock budget
and resumable checkpoints. All distinct exactly reconstructed inequalities are
retained, whether or not they separate the original NS representative and
whether or not they are facets.


In [19]:
"""Construct the exact 64-coordinate target for one ordered pair."""
function perturb_target(target, vertex_j, vertex_k, epsilon::Rational{Int})
    target_weight = 1 - 3 * epsilon / 2
    target_weight >= 0 || error(
        "epsilon must satisfy 1 - 3epsilon/2 >= 0",
    )

    return (
        target_weight .* target .+
        epsilon .* Int.(vertex_j) .+
        (epsilon / 2) .* Int.(vertex_k)
    )
end

function candidate_paths(output_directory, class::Int)
    prefix = "class_" * lpad(string(class), 2, "0")
    return (
        checkpoint=joinpath(output_directory, prefix * "_checkpoint.jls"),
        table=joinpath(output_directory, prefix * "_candidates.tsv"),
        failures=joinpath(output_directory, prefix * "_failures.tsv"),
    )
end

function rational_string(value::Rational)
    denominator(value) == 1 && return string(numerator(value))
    return "$(numerator(value))/$(denominator(value))"
end

optional_rational_string(value) =
    value === nothing ? "" : rational_string(value)

function coefficient_string(coefficients)
    return join(rational_string.(coefficients), ",")
end

function integer_coefficient_string(coefficients)
    return join(string.(coefficients), ",")
end

"""Write the complete candidate table atomically."""
function write_candidate_table(path, candidates)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        println(io, join([
            "source",
            "class",
            "first_j",
            "first_k",
            "beta",
            "original_target_slack",
            "perturbed_target_slack",
            "separates_original",
            "separates_perturbed",
            "original_critical_visibility",
            "original_critical_noise_fraction",
            "perturbed_critical_visibility",
            "perturbed_critical_noise_fraction",
            "cycle_minimum_slack",
            "clockwise_minimum_slack",
            "counterclockwise_minimum_slack",
            "saturation_count",
            "face_dimension",
            "is_facet",
            "solver_objective",
            "coefficients_in_canonical_64_order",
            "primitive_beta",
            "primitive_coefficients_in_canonical_64_order",
        ], "\t"))

        for candidate in candidates
            println(io, join([
                string(candidate.source),
                string(candidate.class),
                string(candidate.first_j),
                string(candidate.first_k),
                rational_string(candidate.exact_beta),
                rational_string(candidate.original_target_slack),
                rational_string(candidate.perturbed_target_slack),
                string(candidate.separates_original),
                string(candidate.separates_perturbed),
                optional_rational_string(candidate.original_critical_visibility),
                optional_rational_string(candidate.original_critical_noise_fraction),
                optional_rational_string(candidate.perturbed_critical_visibility),
                optional_rational_string(candidate.perturbed_critical_noise_fraction),
                rational_string(candidate.exact_cycle_minimum_slack),
                rational_string(candidate.exact_clockwise_minimum_slack),
                rational_string(candidate.exact_counterclockwise_minimum_slack),
                string(candidate.saturation_count),
                string(candidate.face_dimension),
                string(candidate.is_facet),
                @sprintf("%.16g", candidate.solver_objective),
                coefficient_string(candidate.exact_coefficients),
                string(candidate.primitive_beta),
                integer_coefficient_string(candidate.primitive_coefficients),
            ], "\t"))
        end
    end
    mv(temporary_path, path; force=true)
    return path
end

"""Write a checkpoint through a temporary file to avoid partial state files."""
function write_checkpoint(path, state)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        serialize(io, state)
    end
    mv(temporary_path, path; force=true)
    return path
end

function read_checkpoint(path)
    return open(path, "r") do io
        deserialize(io)
    end
end

function log_failure(path, class, pair_j, pair_k, kind, message)
    new_file = !isfile(path)
    safe_message = replace(string(message), '\t' => ' ', '\n' => ' ')
    open(path, "a") do io
        if new_file
            println(io, "timestamp\tclass\tj\tk\tkind\tmessage")
        end
        println(io, join([
            string(now()),
            string(class),
            string(pair_j),
            string(pair_k),
            string(kind),
            safe_message,
        ], "\t"))
    end
end

function next_ordered_pair(j::Int, k::Int, n_vertices::Int)
    next_j = j
    next_k = k + 1
    if next_k > n_vertices
        next_j += 1
        next_k = 1
    end
    return next_j, next_k
end

candidate_key(candidate) = candidate_key(
    candidate.exact_coefficients,
    candidate.exact_beta,
)


candidate_key (generic function with 2 methods)

## 9. Checkpointed search engine

Every new run begins with the already-audited unperturbed candidate. On
resume, the checkpoint is accepted only if its class, perturbation
parameter, vertex count, coefficient representation, and LP type agree
with this notebook.

There is no facet filter. Exact 65-tuples $(c_1,\ldots,c_{64},\beta)$ remove
literal duplicates, while the TSV file preserves all other candidates and
their face information. In particular, gauge-equivalent but numerically
different coefficient tuples may both be saved. Solver, reconstruction, or
audit failures are recorded with their ordered pair and the search
continues. Interrupting the kernel triggers a final checkpoint before the
interrupt is propagated.


In [21]:
function run_class_search!(
    class::Int,
    model,
    coefficients,
    beta,
    original_target,
    baseline_candidate,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    epsilon=1//1000,
    target_seconds=60.0,
    checkpoint_interval_seconds=300.0,
    progress_every=500,
    output_directory=joinpath(
        pwd(),
        "cycle_free_beta_perturbation_results",
    ),
    resume=true,
)
    mkpath(output_directory)
    paths = candidate_paths(output_directory, class)
    n_vertices = size(vertices_cycle, 1)

    if resume && isfile(paths.checkpoint)
        state = read_checkpoint(paths.checkpoint)
        state["format_version"] == 3 || error(
            "Checkpoint format mismatch. This direct 64-coordinate " *
            "notebook cannot resume a checkpoint made by the earlier " *
            "50-coordinate draft.",
        )
        state["lp_kind"] == "free_beta_white_noise" || error(
            "Checkpoint LP-kind mismatch",
        )
        state["coefficient_representation"] ==
            "canonical_probability_64_no_affine_gauge" || error(
            "Checkpoint coefficient-representation mismatch",
        )
        state["class"] == class || error("Checkpoint class mismatch")
        state["epsilon"] == epsilon || error("Checkpoint epsilon mismatch")
        state["n_vertices"] == n_vertices || error(
            "Checkpoint vertex-count mismatch",
        )

        pair_j = state["next_j"]
        pair_k = state["next_k"]
        total_lp_solves = state["total_lp_solves"]
        total_failures = state["total_failures"]
        total_duplicates = state["total_duplicates"]
        elapsed_before_call = state["elapsed_seconds"]
        candidates = state["candidates"]

        println(
            "Resuming class $class at (j,k)=($pair_j,$pair_k) with " *
            "$(length(candidates)) stored coefficient vectors.",
        )
    else
        pair_j = 1
        pair_k = 1
        total_lp_solves = 0
        total_failures = 0
        total_duplicates = 0
        elapsed_before_call = 0.0
        candidates = Any[baseline_candidate]
    end

    seen_exact = Set{Any}(candidate_key(candidate) for candidate in candidates)
    candidate_key(baseline_candidate) in seen_exact || error(
        "The unperturbed baseline is missing from the checkpoint",
    )

    remaining_seconds = max(0.0, target_seconds - elapsed_before_call)
    if remaining_seconds == 0.0 || pair_j > n_vertices
        println(
            "Class $class already reached its requested budget or exhausted all pairs.",
        )
        write_candidate_table(paths.table, candidates)
        return candidates
    end

    started_at = time()
    last_checkpoint_at = started_at

    elapsed_total() = elapsed_before_call + (time() - started_at)

    function current_state()
        return Dict{String, Any}(
            "format_version" => 3,
            "lp_kind" => "free_beta_white_noise",
            "coefficient_representation" =>
                "canonical_probability_64_no_affine_gauge",
            "class" => class,
            "epsilon" => epsilon,
            "n_vertices" => n_vertices,
            "next_j" => pair_j,
            "next_k" => pair_k,
            "total_lp_solves" => total_lp_solves,
            "total_failures" => total_failures,
            "total_duplicates" => total_duplicates,
            "elapsed_seconds" => min(elapsed_total(), target_seconds),
            "candidates" => candidates,
            "saved_at" => string(now()),
        )
    end

    try
        while pair_j <= n_vertices && elapsed_total() < target_seconds
            if pair_j == pair_k
                pair_j, pair_k = next_ordered_pair(
                    pair_j,
                    pair_k,
                    n_vertices,
                )
                continue
            end

            current_j = pair_j
            current_k = pair_k

            try
                perturbed_target = perturb_target(
                    original_target,
                    view(vertices_cycle, current_j, :),
                    view(vertices_cycle, current_k, :),
                    epsilon,
                )

                numerical_solution, status, solver_objective =
                    solve_free_beta_target!(
                        model,
                        coefficients,
                        beta,
                        perturbed_target,
                    )
                total_lp_solves += 1

                if numerical_solution === nothing
                    total_failures += 1
                    log_failure(
                        paths.failures,
                        class,
                        current_j,
                        current_k,
                        :solver_status,
                        status,
                    )
                else
                    reconstructed = reconstruct_free_beta_solution(
                        numerical_solution.coefficients,
                        numerical_solution.beta,
                    )

                    if reconstructed === nothing
                        total_failures += 1
                        log_failure(
                            paths.failures,
                            class,
                            current_j,
                            current_k,
                            :rational_reconstruction,
                            "Could not reconstruct a positive normalized solution",
                        )
                    else
                        exact_key = candidate_key(
                            reconstructed.coefficients,
                            reconstructed.beta,
                        )

                        if exact_key in seen_exact
                            total_duplicates += 1
                        else
                            candidate = audit_free_beta_candidate(
                                :ordered_pair,
                                class,
                                current_j,
                                current_k,
                                reconstructed,
                                solver_objective,
                                original_target,
                                perturbed_target,
                                vertices_cycle,
                                vertices_clockwise,
                                vertices_counterclockwise,
                            )

                            push!(seen_exact, exact_key)
                            push!(candidates, candidate)

                            label = candidate.is_facet ? "FACET" : "nonfacet"
                            println(
                                "Class $class: new $label from " *
                                "(j,k)=($current_j,$current_k), " *
                                "face dimension=$(candidate.face_dimension), " *
                                "original slack=$(candidate.original_target_slack), " *
                                "perturbed slack=$(candidate.perturbed_target_slack)",
                            )
                        end
                    end
                end
            catch error_value
                error_value isa InterruptException && rethrow()
                total_failures += 1
                log_failure(
                    paths.failures,
                    class,
                    current_j,
                    current_k,
                    :exception,
                    sprint(showerror, error_value),
                )
                @warn "Error while processing ordered pair" class current_j current_k exception=(error_value, catch_backtrace())
            end

            pair_j, pair_k = next_ordered_pair(
                pair_j,
                pair_k,
                n_vertices,
            )

            if total_lp_solves > 0 && total_lp_solves % progress_every == 0
                facet_count = count(candidate -> candidate.is_facet, candidates)
                @printf(
                    "Class %d | next=(%d,%d) | LPs=%d | stored=%d | facets=%d | exact duplicates=%d | failures=%d | elapsed=%.1f min\n",
                    class,
                    pair_j,
                    pair_k,
                    total_lp_solves,
                    length(candidates),
                    facet_count,
                    total_duplicates,
                    total_failures,
                    elapsed_total() / 60,
                )
            end

            if time() - last_checkpoint_at >= checkpoint_interval_seconds
                write_checkpoint(paths.checkpoint, current_state())
                write_candidate_table(paths.table, candidates)
                last_checkpoint_at = time()
            end
        end
    finally
        write_checkpoint(paths.checkpoint, current_state())
        write_candidate_table(paths.table, candidates)
    end

    facet_count = count(candidate -> candidate.is_facet, candidates)
    println(
        "Finished current budget for class $class: " *
        "$(length(candidates)) stored coefficient vectors, " *
        "$facet_count facets, $total_lp_solves total LP solves, " *
        "$total_failures failures.",
    )
    println("Results directory: ", abspath(output_directory))

    return candidates
end


run_class_search! (generic function with 1 method)

## 10. Run configuration

The default is a 60-second installation test per class. After verifying the
output, change only `TARGET_SECONDS_PER_CLASS`. Examples:

- `3600.0`: one cumulative hour per class;
- `7200.0`: two cumulative hours per class;
- `10800.0`: three cumulative hours per class.

The time is cumulative within each checkpoint. Thus, increasing the value
from 60 to 3600 seconds continues for approximately another 3540 seconds
rather than starting again.

Checkpoints created by the discarded 50-coordinate draft are intentionally
incompatible with this direct 64-coordinate version. If such a checkpoint
exists, use a fresh output directory or remove that test output before
starting this notebook.


In [23]:
# -------------------------- USER CONFIGURATION --------------------------

CLASSES_TO_SEARCH = [29]
PERTURBATION_EPSILON = 1//100
TARGET_SECONDS_PER_CLASS = 18_000.0
CHECKPOINT_INTERVAL_SECONDS = 300.0
PROGRESS_EVERY = 500
SEARCH_OUTPUT_DIRECTORY = joinpath(
    pwd(),
    "cycle_free_beta_perturbation_results_eps_1e-2",
)
RESUME_SEARCH = true

# ------------------------------------------------------------------------

perturbation_results = Dict{Int, Any}()

println("Free-beta perturbation results will be written to:")
println(abspath(SEARCH_OUTPUT_DIRECTORY))

for class in CLASSES_TO_SEARCH
    println("\n", repeat("=", 72))
    println("Starting or resuming free-beta search for class $class")
    println(repeat("=", 72))

    perturbation_results[class] = run_class_search!(
        class,
        free_beta_model,
        free_beta_coefficients,
        free_beta_beta,
        NS_TARGETS[class],
        BASELINE_RESULTS[class],
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise;
        epsilon=PERTURBATION_EPSILON,
        target_seconds=TARGET_SECONDS_PER_CLASS,
        checkpoint_interval_seconds=CHECKPOINT_INTERVAL_SECONDS,
        progress_every=PROGRESS_EVERY,
        output_directory=SEARCH_OUTPUT_DIRECTORY,
        resume=RESUME_SEARCH,
    )
end

println("\nSearch summary")
println("class  stored coefficient vectors  facets  separating original target")
for class in CLASSES_TO_SEARCH
    candidates = perturbation_results[class]
    @printf(
        "%5d  %26d  %6d  %26d\n",
        class,
        length(candidates),
        count(candidate -> candidate.is_facet, candidates),
        count(candidate -> candidate.separates_original, candidates),
    )
end


Free-beta perturbation results will be written to:
C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_free_beta_perturbation_results_eps_1e-2

Starting or resuming free-beta search for class 29
Resuming class 29 at (j,k)=(17,2891) with 4 stored coefficient vectors.
Class 29 | next=(17,2970) | LPs=133000 | stored=4 | facets=4 | exact duplicates=132997 | failures=0 | elapsed=15.0 min
Class 29 | next=(17,3470) | LPs=133500 | stored=4 | facets=4 | exact duplicates=133497 | failures=0 | elapsed=15.1 min
Class 29 | next=(17,3970) | LPs=134000 | stored=4 | facets=4 | exact duplicates=133997 | failures=0 | elapsed=15.2 min
Class 29 | next=(17,4470) | LPs=134500 | stored=4 | facets=4 | exact duplicates=134497 | failures=0 | elapsed=15.2 min
Class 29 | next=(17,4970) | LPs=135000 | stored=4 | facets=4 | exact duplicates=134997 | failures=0 | elapsed=15.3 min
Class 29 | next=(17,5470) 

Excessive output truncated after 524322 bytes.

## 11. Reproducibility metadata and checkpoint diagnostics

The final cell records the Julia version, package status, LP definition,
coefficient representation, search parameters, and current checkpoint
positions. Send the four candidate TSV files, four checkpoints, failure
files if any, and `run_metadata.txt` for an independent audit.


In [25]:
using InteractiveUtils

versioninfo()
Pkg.status()

mkpath(SEARCH_OUTPUT_DIRECTORY)
metadata_path = joinpath(SEARCH_OUTPUT_DIRECTORY, "run_metadata.txt")

open(metadata_path, "w") do io
    println(io, "created_at = ", now())
    println(io, "julia_version = ", VERSION)
    println(io, "lp_kind = free_beta_white_noise")
    println(io, "coefficient_representation = canonical_probability_64_no_affine_gauge")
    println(io, "deduplication = exact reconstructed (64 coefficients, beta) tuple")
    println(io, "gauge_equivalent_duplicates_may_remain = true")
    println(io, "coordinate_order = xyz blocks and abc columns in 000,001,010,011,100,101,110,111 order")
    println(io, "siglocal_vertices = ", size(vertices_cycle, 1))
    println(io, "siglocal_affine_dimension = ", SIGLOCAL_AFFINE_DIMENSION)
    println(io, "classes = ", join(CLASSES_TO_SEARCH, ","))
    println(io, "epsilon = ", PERTURBATION_EPSILON)
    println(io, "target_seconds_per_class = ", TARGET_SECONDS_PER_CLASS)
    println(io, "checkpoint_interval_seconds = ", CHECKPOINT_INTERVAL_SECONDS)
    println(io, "progress_every = ", PROGRESS_EVERY)
    println(io, "resume = ", RESUME_SEARCH)
    for class in CLASSES
        println(
            io,
            "baseline_slack_class_", class, " = ",
            BASELINE_RESULTS[class].original_target_slack,
        )
    end
end

println("\nCheckpoint diagnostics")
for class in CLASSES_TO_SEARCH
    paths = candidate_paths(SEARCH_OUTPUT_DIRECTORY, class)
    if isfile(paths.checkpoint)
        state = read_checkpoint(paths.checkpoint)
        println(
            "class = ", class,
            ", LP solves = ", state["total_lp_solves"],
            ", failures = ", state["total_failures"],
            ", exact duplicates = ", state["total_duplicates"],
            ", elapsed seconds = ", state["elapsed_seconds"],
            ", next pair = (", state["next_j"], ", ", state["next_k"], ")",
            ", stored coefficient vectors = ", length(state["candidates"]),
        )
    else
        println("class = ", class, ": no checkpoint written")
    end
end

println("Metadata written to: ", abspath(metadata_path))


Julia Version 1.12.6
Commit 15346901f0 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Windows (x86_64-w64-mingw32)
  CPU: 12 × 12th Gen Intel(R) Core(TM) i5-12450HX
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, alderlake)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 12 virtual cores)
Status `C:\Users\josen\.julia\environments\v1.12\Project.toml`
  [60bf3e95] GLPK v1.2.1
  [7073ff75] IJulia v1.34.4
⌃ [4076af6c] JuMP v1.30.1
  [b8f27783] MathOptInterface v1.54.0
  [92933f4c] ProgressMeter v1.11.0
  [5cd12e4b] QuantumNPA v0.1.2
  [c946c3f1] SCS v2.7.0
Info Packages marked with ⌃ have new versions available and may be upgradable.

Checkpoint diagnostics
class = 29, LP solves = 2627579, failures = 0, exact duplicates = 2627576, elapsed seconds = 18000.0, next pair = (324, 2560), stored coefficient vectors = 4
Metadata written to: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_pr